**Install Libraries**

Let's install all the necessary libraries required for this notebook. We'll use `pip install` to get them. This ensures that we have all the tools we need before running the code.

In [23]:
!pip install requests pandas nltk scikit-learn


[notice] A new release of pip is available: 24.3.1 -> 25.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:
import numpy as np
import random

# Set seeds for reproducibility
np.random.seed(42)
random.seed(42)

print("Seeds are set to 42.")

Seeds are set to 42.


# Binary Sentiment Classification 

Welcome to sentiment classification project! In this notebook, we'll walk through the entire process of building a model to classify movie reviews as positive or negative. We'll start by loading the data, exploring it to understand its characteristics, preprocessing the text to make it suitable for machine learning, and finally, building and evaluating different models to find the best one for our task.

Let's get started!

## 1. Data Loading

First things first, we need to load our datasets. We'll define a function to download the data from the URLs and load it into pandas DataFrames. This is a crucial first step as it makes our data accessible for further analysis and processing.  We're using `requests` to fetch the zip files and `zipfile` and `io` to handle the zip archive in memory without saving it to disk first. Pandas `read_csv` will then load the CSV data into a DataFrame. This approach is efficient and keeps our workspace clean. Let's do it!

In [25]:
import requests
import zipfile
import io
import pandas as pd

def load_data():
    train_url = 'https://static.cdn.epam.com/uploads/583f9e4a37492715074c531dbd5abad2/ds/final_project_train_dataset.zip'
    test_url = 'https://static.cdn.epam.com/uploads/583f9e4a37492715074c531dbd5abad2/ds/final_project_test_dataset.zip'

    train_response = requests.get(train_url)
    train_zip_file = zipfile.ZipFile(io.BytesIO(train_response.content))
    train_data = pd.read_csv(train_zip_file.open('final_project_train_dataset/train.csv'))

    test_response = requests.get(test_url)
    test_zip_file = zipfile.ZipFile(io.BytesIO(test_response.content))
    test_data = pd.read_csv(test_zip_file.open('final_project_test_dataset/test.csv'))

    return train_data, test_data

train_df, test_df = load_data()

print("Train data loaded successfully!")
print("Test data loaded successfully!")

Train data loaded successfully!
Test data loaded successfully!


## 2. Exploratory Data Analysis (EDA)

Now that we have loaded our data, it's time to get to know it better. EDA is essential to understand the data's structure, identify potential issues, and gain insights that can guide our preprocessing and modeling strategies. We'll start by looking at the first few rows, checking for missing values, and examining the distribution of the target variable ('sentiment'). This step will help us make informed decisions down the line. 

In [26]:
print("\nTrain Data Head:")
print(train_df.head())

print("\nTrain Data Info:")
print(train_df.info())

print("\nTrain Data Describe:")
print(train_df.describe())

print("\nSentiment Distribution in Train Data:")
print(train_df['sentiment'].value_counts(normalize=True))

print("\nMissing values in Train Data:")
print(train_df.isnull().sum())

print("\nTest Data Head:")
print(test_df.head())

print("\nTest Data Info:")
print(test_df.info())

print("\nTest Data Describe:")
print(test_df.describe())

print("\nSentiment Distribution in Test Data:")
print(test_df['sentiment'].value_counts(normalize=True))

print("\nMissing values in Test Data:")
print(test_df.isnull().sum())


Train Data Head:
                                              review sentiment
0  I caught this little gem totally by accident b...  positive
1  I can't believe that I let myself into this mo...  negative
2  *spoiler alert!* it just gets to me the nerve ...  negative
3  If there's one thing I've learnt from watching...  negative
4  I remember when this was in theaters, reviews ...  negative

Train Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     40000 non-null  object
 1   sentiment  40000 non-null  object
dtypes: object(2)
memory usage: 625.1+ KB
None

Train Data Describe:
                                                   review sentiment
count                                               40000     40000
unique                                              39728         2
top     Loved today's show!!! It was a variety an

### EDA Insights and Conclusions

*   **Data Overview**: Both train and test datasets contain 'review' text and 'sentiment' labels. The 'sentiment' is our target variable for binary classification.
*   **Data Types**: The 'review' column is of object type (string), and 'sentiment' is of object type (string). We might keep it as is.
*   **Target Distribution**: The sentiment distribution in both train and test sets is perfectly balanced (50% positive and 50% negative). This is great news as we don't have to worry about class imbalance issues, and accuracy will be a reliable metric.
*   **Missing Values**:  No missing values in either the train or test datasets. This simplifies our preprocessing as we don't need to handle imputation.
*   **Text Length**: From `describe()`, we can see some basic stats on review text, though it's not directly numerical. We might want to explore text length distribution later, but for now, it's not critical.

**Conclusion from EDA:**

The datasets are clean and balanced, which is a fantastic starting point. We can proceed with text preprocessing, focusing on cleaning and transforming the 'review' text into a numerical format suitable for machine learning models. No immediate concerns or issues were found during EDA. Let's move on to text preprocessing!

## 3. Text Preprocessing

Text preprocessing is a crucial step in NLP. Raw text data is often messy and needs to be transformed into a more digestible format for machine learning models. Here, we will implement several common preprocessing techniques:

1.  **Tokenization**: Breaking down the text into individual words or tokens.
2.  **Stop-word Removal**: Filtering out common words that don't carry much meaning (e.g., 'the', 'is', 'and').
3.  **Stemming vs Lemmatization**: Reducing words to their root form. We'll compare stemming and lemmatization to see which works better for our task.
4.  **Vectorization**: Converting the preprocessed text into numerical vectors. We'll explore at least two vectorization techniques.

Let's start with tokenization and move through each step, justifying our choices and observing the effects.

### 3.1. Tokenization

Tokenization is the process of splitting text into individual units, or tokens.  For our task, word tokenization seems appropriate, where we break down each review into a list of words. We'll use NLTK's `word_tokenize` for this, as it's a robust and widely used tokenizer in NLP. Tokenization is the foundation for further text processing steps, as we need to work with individual words to remove stop words, stem/lemmatize, and vectorize. Let's tokenize our reviews!

In [27]:
import nltk
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
import nltk.data # Import nltk.data to manage data paths

nltk_data_path = 'nltk_data' # Using a generic path for nltk_data
nltk.data.path.append(nltk_data_path) # Add path to NLTK's data path list

try:
    nltk.data.find('tokenizers/punkt')
    print("punkt tokenizer data found.")
except LookupError:
    nltk.download('punkt', download_dir=nltk_data_path) 

try:
    nltk.data.find('tokenizers/punkt_tab')
    print("punkt_tab tokenizer data found.")
except LookupError:
    nltk.download('punkt_tab', download_dir=nltk_data_path) 

try:
    nltk.data.find('corpora/stopwords')
    print("stopwords corpus data found.")
except LookupError:
    nltk.download('stopwords', download_dir=nltk_data_path) 

try:
    nltk.data.find('corpora/wordnet')
    print("wordnet corpus data found.")
except LookupError:
    nltk.download('wordnet', download_dir=nltk_data_path) 


def tokenize_text(text):
    return word_tokenize(text.lower()) # lowercase for consistency

train_df['tokenized_review'] = train_df['review'].apply(tokenize_text)
test_df['tokenized_review'] = test_df['review'].apply(tokenize_text)

print("Tokenization completed!")
print("\nSample tokenized review:")
print(train_df['tokenized_review'].head())

punkt tokenizer data found.


[nltk_data] Downloading package punkt_tab to nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package wordnet to nltk_data...


stopwords corpus data found.
Tokenization completed!

Sample tokenized review:
0    [i, caught, this, little, gem, totally, by, ac...
1    [i, ca, n't, believe, that, i, let, myself, in...
2    [*, spoiler, alert, !, *, it, just, gets, to, ...
3    [if, there, 's, one, thing, i, 've, learnt, fr...
4    [i, remember, when, this, was, in, theaters, ,...
Name: tokenized_review, dtype: object


### 3.2. Stop-word Removal

Stop words are common words in a language that are often filtered out in NLP tasks because they don't usually contribute significantly to the meaning of the text. Examples include 'a', 'an', 'the', 'is', etc. Removing stop words can reduce noise and the dimensionality of our data, potentially improving model performance and efficiency. We'll use NLTK's list of English stop words and remove them from our tokenized reviews. Let's remove those stop words!

In [28]:
stop_words = set(stopwords.words('english'))

def remove_stopwords(tokens):
    return [token for token in tokens if token not in stop_words and token not in string.punctuation] # also remove punctuation

train_df['review_without_stopwords'] = train_df['tokenized_review'].apply(remove_stopwords)
test_df['review_without_stopwords'] = test_df['tokenized_review'].apply(remove_stopwords)

print("Stop-word removal completed!")
print("\nSample review without stop words:")
print(train_df['review_without_stopwords'].head())

Stop-word removal completed!

Sample review without stop words:
0    [caught, little, gem, totally, accident, back,...
1    [ca, n't, believe, let, movie, accomplish, fav...
2    [spoiler, alert, gets, nerve, people, remake, ...
3    ['s, one, thing, 've, learnt, watching, george...
4    [remember, theaters, reviews, said, horrible, ...
Name: review_without_stopwords, dtype: object


### 3.3. Stemming vs. Lemmatization

Stemming and lemmatization are techniques to reduce words to their root form, which helps in grouping together different forms of the same word. 

*   **Stemming** is a simpler approach that chops off the ends of words based on heuristics. It's faster but can be less accurate and might produce stems that are not actual words.
*   **Lemmatization** is more sophisticated and uses vocabulary and morphological analysis to return the base or dictionary form of a word (lemma). It's generally more accurate but computationally more intensive.

We will apply both stemming (using Porter Stemmer) and lemmatization (using WordNet Lemmatizer) to our reviews and compare their effects. We'll then decide which one to use in our final preprocessing pipeline based on model performance. Let's see how they compare!

In [29]:
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def stem_tokens(tokens):
    return [stemmer.stem(token) for token in tokens]

def lemmatize_tokens(tokens):
    return [lemmatizer.lemmatize(token) for token in tokens]

train_df['stemmed_review'] = train_df['review_without_stopwords'].apply(stem_tokens)
test_df['stemmed_review'] = test_df['review_without_stopwords'].apply(stem_tokens)

train_df['lemmatized_review'] = train_df['review_without_stopwords'].apply(lemmatize_tokens)
test_df['lemmatized_review'] = test_df['review_without_stopwords'].apply(lemmatize_tokens)

print("Stemming and Lemmatization completed!")
print("\nSample stemmed review:")
print(train_df['stemmed_review'].head())
print("\nSample lemmatized review:")
print(train_df['lemmatized_review'].head())

Stemming and Lemmatization completed!

Sample stemmed review:
0    [caught, littl, gem, total, accid, back, 1980,...
1    [ca, n't, believ, let, movi, accomplish, favor...
2    [spoiler, alert, get, nerv, peopl, remak, use,...
3    ['s, one, thing, 've, learnt, watch, georg, ro...
4    [rememb, theater, review, said, horribl, well,...
Name: stemmed_review, dtype: object

Sample lemmatized review:
0    [caught, little, gem, totally, accident, back,...
1    [ca, n't, believe, let, movie, accomplish, fav...
2    [spoiler, alert, get, nerve, people, remake, u...
3    ['s, one, thing, 've, learnt, watching, george...
4    [remember, theater, review, said, horrible, we...
Name: lemmatized_review, dtype: object


### 3.4. Vectorization

Now that we have preprocessed our text data, we need to convert it into a numerical format that machine learning models can understand. This process is called vectorization. We will explore two common techniques:

1.  **Bag of Words (CountVectorizer)**: This method creates a vocabulary of all unique words in the corpus and represents each document as a vector where each element counts the frequency of words in the document. 
2.  **TF-IDF (TfidfVectorizer)**: Term Frequency-Inverse Document Frequency. This method not only counts word frequencies but also weighs them down if they are common across all documents, emphasizing words that are important to specific documents in the corpus.

We will vectorize the stemmed and lemmatized reviews separately using both CountVectorizer and TfidfVectorizer. This will allow us to test different combinations of preprocessing and vectorization techniques to find the most effective approach for our sentiment classification task. Let's vectorize!

In [30]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

def dummy_fun(doc):
    return doc

# Vectorize stemmed reviews
count_vectorizer_stemmed = CountVectorizer(tokenizer=dummy_fun, preprocessor=dummy_fun, lowercase=False, token_pattern=None)
tfidf_vectorizer_stemmed = TfidfVectorizer(tokenizer=dummy_fun, preprocessor=dummy_fun, lowercase=False, token_pattern=None)

X_train_count_stemmed = count_vectorizer_stemmed.fit_transform(train_df['stemmed_review'])
X_test_count_stemmed = count_vectorizer_stemmed.transform(test_df['stemmed_review'])

X_train_tfidf_stemmed = tfidf_vectorizer_stemmed.fit_transform(train_df['stemmed_review'])
X_test_tfidf_stemmed = tfidf_vectorizer_stemmed.transform(test_df['stemmed_review'])

# Vectorize lemmatized reviews
count_vectorizer_lemmatized = CountVectorizer(tokenizer=dummy_fun, preprocessor=dummy_fun, lowercase=False, token_pattern=None)
tfidf_vectorizer_lemmatized = TfidfVectorizer(tokenizer=dummy_fun, preprocessor=dummy_fun, lowercase=False, token_pattern=None)

X_train_count_lemmatized = count_vectorizer_lemmatized.fit_transform(train_df['lemmatized_review'])
X_test_count_lemmatized = count_vectorizer_lemmatized.transform(test_df['lemmatized_review'])

X_train_tfidf_lemmatized = tfidf_vectorizer_lemmatized.fit_transform(train_df['lemmatized_review'])
X_test_tfidf_lemmatized = tfidf_vectorizer_lemmatized.transform(test_df['lemmatized_review'])

y_train = train_df['sentiment']
y_test = test_df['sentiment']

print("Vectorization completed!")
print("\nShape of Count vectorized stemmed training data:", X_train_count_stemmed.shape)
print("Shape of TF-IDF vectorized stemmed training data:", X_train_tfidf_stemmed.shape)
print("Shape of Count vectorized lemmatized training data:", X_train_count_lemmatized.shape)
print("Shape of TF-IDF vectorized lemmatized training data:", X_train_tfidf_lemmatized.shape)

Vectorization completed!

Shape of Count vectorized stemmed training data: (40000, 115604)
Shape of TF-IDF vectorized stemmed training data: (40000, 115604)
Shape of Count vectorized lemmatized training data: (40000, 136282)
Shape of TF-IDF vectorized lemmatized training data: (40000, 136282)


## 4. Modeling

With our text data preprocessed and vectorized, we can now move on to building machine learning models for sentiment classification. We will start with a baseline model and then explore a few other models to find the best performer. We need to test at least 3 models. For a binary classification task like sentiment analysis, good starting points include:

1.  **Logistic Regression**: A simple and effective linear model for binary classification. It's often a good baseline for text classification tasks.
2.  **Support Vector Machines (SVM)**: Powerful and effective in high-dimensional spaces, which is common in text data after vectorization.
3.  **Naive Bayes (MultinomialNB)**: Particularly well-suited for text classification with discrete features (like word counts). It's computationally efficient and often performs surprisingly well.

We will train and evaluate each of these models using different vectorized datasets (stemmed/lemmatized with CountVectorizer and TfidfVectorizer). Our evaluation metric is accuracy, as specified in the task, and we aim to achieve at least 0.85 accuracy on the test set. Let's start building and training our models!

### 4.1. Baseline Model - Logistic Regression

Logistic Regression is a solid baseline model for binary classification. It's interpretable and often performs well with text data. We'll train Logistic Regression models on all four vectorized datasets (CountVectorizer and TfidfVectorizer with both stemmed and lemmatized text) to establish a baseline performance and see which preprocessing and vectorization combination initially looks promising. Let's train!

In [31]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

lr_model = LogisticRegression(max_iter=1000, random_state=42)

print("\nLogistic Regression - CountVectorizer Stemmed")
lr_model.fit(X_train_count_stemmed, y_train)
y_pred_lr_count_stemmed = lr_model.predict(X_test_count_stemmed)
accuracy_lr_count_stemmed = accuracy_score(y_test, y_pred_lr_count_stemmed)
print(f"Accuracy: {accuracy_lr_count_stemmed:.4f}")

print("\nLogistic Regression - TfidfVectorizer Stemmed")
lr_model.fit(X_train_tfidf_stemmed, y_train)
y_pred_lr_tfidf_stemmed = lr_model.predict(X_test_tfidf_stemmed)
accuracy_lr_tfidf_stemmed = accuracy_score(y_test, y_pred_lr_tfidf_stemmed)
print(f"Accuracy: {accuracy_lr_tfidf_stemmed:.4f}")

print("\nLogistic Regression - CountVectorizer Lemmatized")
lr_model.fit(X_train_count_lemmatized, y_train)
y_pred_lr_count_lemmatized = lr_model.predict(X_test_count_lemmatized)
accuracy_lr_count_lemmatized = accuracy_score(y_test, y_pred_lr_count_lemmatized)
print(f"Accuracy: {accuracy_lr_count_lemmatized:.4f}")

print("\nLogistic Regression - TfidfVectorizer Lemmatized")
lr_model.fit(X_train_tfidf_lemmatized, y_train)
y_pred_lr_tfidf_lemmatized = lr_model.predict(X_test_tfidf_lemmatized)
accuracy_lr_tfidf_lemmatized = accuracy_score(y_test, y_pred_lr_tfidf_lemmatized)
print(f"Accuracy: {accuracy_lr_tfidf_lemmatized:.4f}")


Logistic Regression - CountVectorizer Stemmed
Accuracy: 0.8871

Logistic Regression - TfidfVectorizer Stemmed
Accuracy: 0.8970

Logistic Regression - CountVectorizer Lemmatized
Accuracy: 0.8913

Logistic Regression - TfidfVectorizer Lemmatized
Accuracy: 0.8972


### 4.2. Support Vector Machine (SVM)

Support Vector Machines (SVMs) are known for their effectiveness in high-dimensional spaces and are often a strong choice for text classification. We'll train SVM models using the same four vectorized datasets to compare their performance against Logistic Regression and to see if SVMs can push our accuracy higher. Let's train some SVMs!

In [32]:
from sklearn.svm import LinearSVC

svm_model = LinearSVC(max_iter=2000, random_state=42) 
# 2000 iterations is good enough; increasing iterations does not improve accuracy so ignore warning.

print("\nSVM - CountVectorizer Stemmed")
svm_model.fit(X_train_count_stemmed, y_train)
y_pred_svm_count_stemmed = svm_model.predict(X_test_count_stemmed)
accuracy_svm_count_stemmed = accuracy_score(y_test, y_pred_svm_count_stemmed)
print(f"Accuracy: {accuracy_svm_count_stemmed:.4f}")

print("\nSVM - TfidfVectorizer Stemmed")
svm_model.fit(X_train_tfidf_stemmed, y_train)
y_pred_svm_tfidf_stemmed = svm_model.predict(X_test_tfidf_stemmed)
accuracy_svm_tfidf_stemmed = accuracy_score(y_test, y_pred_svm_tfidf_stemmed)
print(f"Accuracy: {accuracy_svm_tfidf_stemmed:.4f}")

print("\nSVM - CountVectorizer Lemmatized")
svm_model.fit(X_train_count_lemmatized, y_train)
y_pred_svm_count_lemmatized = svm_model.predict(X_test_count_lemmatized)
accuracy_svm_count_lemmatized = accuracy_score(y_test, y_pred_svm_count_lemmatized)
print(f"Accuracy: {accuracy_svm_count_lemmatized:.4f}")

print("\nSVM - TfidfVectorizer Lemmatized")
svm_model.fit(X_train_tfidf_lemmatized, y_train)
y_pred_svm_tfidf_lemmatized = svm_model.predict(X_test_tfidf_lemmatized)
accuracy_svm_tfidf_lemmatized = accuracy_score(y_test, y_pred_svm_tfidf_lemmatized)
print(f"Accuracy: {accuracy_svm_tfidf_lemmatized:.4f}")


SVM - CountVectorizer Stemmed


c:\Users\maksy\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Accuracy: 0.8680

SVM - TfidfVectorizer Stemmed
Accuracy: 0.8973

SVM - CountVectorizer Lemmatized


c:\Users\maksy\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Accuracy: 0.8748

SVM - TfidfVectorizer Lemmatized
Accuracy: 0.9033


### 4.3. Naive Bayes - MultinomialNB

Multinomial Naive Bayes is a probabilistic learning method often used in NLP, especially for text classification. It assumes that the features (word counts) are independent, which is a naive assumption but often works well in practice. MultinomialNB is particularly suitable for CountVectorizer outputs because it works with discrete feature counts.  We'll apply MultinomialNB to our CountVectorizer datasets (stemmed and lemmatized) and see how it performs. Let's train!

In [33]:
from sklearn.naive_bayes import MultinomialNB

nb_model = MultinomialNB()

print("\nMultinomialNB - CountVectorizer Stemmed")
nb_model.fit(X_train_count_stemmed, y_train)
y_pred_nb_count_stemmed = nb_model.predict(X_test_count_stemmed)
accuracy_nb_count_stemmed = accuracy_score(y_test, y_pred_nb_count_stemmed)
print(f"Accuracy: {accuracy_nb_count_stemmed:.4f}")

print("\nMultinomialNB - CountVectorizer Lemmatized")
nb_model.fit(X_train_count_lemmatized, y_train)
y_pred_nb_count_lemmatized = nb_model.predict(X_test_count_lemmatized)
accuracy_nb_count_lemmatized = accuracy_score(y_test, y_pred_nb_count_lemmatized)
print(f"Accuracy: {accuracy_nb_count_lemmatized:.4f}")

print("\nMultinomialNB - TfidfVectorizer Stemmed")
nb_model.fit(X_train_tfidf_stemmed, y_train)
y_pred_nb_tfidf_stemmed = nb_model.predict(X_test_tfidf_stemmed)
accuracy_nb_tfidf_stemmed = accuracy_score(y_test, y_pred_nb_tfidf_stemmed)
print(f"Accuracy: {accuracy_nb_tfidf_stemmed:.4f}")

print("\nMultinomialNB - TfidfVectorizer Lemmatized")
nb_model.fit(X_train_tfidf_lemmatized, y_train)
y_pred_nb_tfidf_lemmatized = nb_model.predict(X_test_tfidf_lemmatized)
accuracy_nb_tfidf_lemmatized = accuracy_score(y_test, y_pred_nb_tfidf_lemmatized)
print(f"Accuracy: {accuracy_nb_tfidf_lemmatized:.4f}")


MultinomialNB - CountVectorizer Stemmed
Accuracy: 0.8550

MultinomialNB - CountVectorizer Lemmatized
Accuracy: 0.8582

MultinomialNB - TfidfVectorizer Stemmed
Accuracy: 0.8624

MultinomialNB - TfidfVectorizer Lemmatized
Accuracy: 0.8692


### 4.4. Model Comparison and Selection

We've trained three different models (Logistic Regression, SVM, and MultinomialNB) with different preprocessing and vectorization approaches. Let's summarize the accuracies we achieved to compare them and select the best model configuration for our task.

In [34]:
# Accuracy values from previously defined variables
comparison_data = [
    ["Logistic Regression", "CountVectorizer", "Stemmed", accuracy_lr_count_stemmed],
    ["Logistic Regression", "TfidfVectorizer", "Stemmed", accuracy_lr_tfidf_stemmed],
    ["Logistic Regression", "CountVectorizer", "Lemmatized", accuracy_lr_count_lemmatized],
    ["Logistic Regression", "TfidfVectorizer", "Lemmatized", accuracy_lr_tfidf_lemmatized],
    ["SVM", "CountVectorizer", "Stemmed", accuracy_svm_count_stemmed],
    ["SVM", "TfidfVectorizer", "Stemmed", accuracy_svm_tfidf_stemmed],
    ["SVM", "CountVectorizer", "Lemmatized", accuracy_svm_count_lemmatized],
    ["SVM", "TfidfVectorizer", "Lemmatized", accuracy_svm_tfidf_lemmatized],
    ["MultinomialNB", "CountVectorizer", "Stemmed", accuracy_nb_count_stemmed],
    ["MultinomialNB", "CountVectorizer", "Lemmatized", accuracy_nb_count_lemmatized],
    ["MultinomialNB", "TfidfVectorizer", "Stemmed", accuracy_nb_tfidf_stemmed],
    ["MultinomialNB", "TfidfVectorizer", "Lemmatized", accuracy_nb_tfidf_lemmatized]
]

df_comparison = pd.DataFrame(
    comparison_data,
    columns=["Model", "Vectorizer", "Preprocessing", "Accuracy"]
)

print("Summary of Accuracies:")
print(df_comparison)

Summary of Accuracies:
                  Model       Vectorizer Preprocessing  Accuracy
0   Logistic Regression  CountVectorizer       Stemmed    0.8871
1   Logistic Regression  TfidfVectorizer       Stemmed    0.8970
2   Logistic Regression  CountVectorizer    Lemmatized    0.8913
3   Logistic Regression  TfidfVectorizer    Lemmatized    0.8972
4                   SVM  CountVectorizer       Stemmed    0.8680
5                   SVM  TfidfVectorizer       Stemmed    0.8973
6                   SVM  CountVectorizer    Lemmatized    0.8748
7                   SVM  TfidfVectorizer    Lemmatized    0.9033
8         MultinomialNB  CountVectorizer       Stemmed    0.8550
9         MultinomialNB  CountVectorizer    Lemmatized    0.8582
10        MultinomialNB  TfidfVectorizer       Stemmed    0.8624
11        MultinomialNB  TfidfVectorizer    Lemmatized    0.8692


**Observations and Model Selection:**

*   **Best Performance**: Based on the results, **SVM with TfidfVectorizer on lemmatized text** achieved the highest accuracy. It comfortably exceeds our target accuracy of 0.85.
*   **Vectorizer Impact**: TfidfVectorizer generally seems to perform slightly better than CountVectorizer across models, especially for Logistic Regression and SVM. This suggests that weighting words by their importance in the document and across the corpus (as TF-IDF does) is beneficial for sentiment classification.
*   **Preprocessing Impact**:  Lemmatization seems to offer a slight edge over stemming, particularly with SVM and Logistic Regression, although the difference is not dramatic. For MultinomialNB the performance is also quite good.
*   **Model Choice**: SVM and Logistic Regression both perform strongly, with SVM slightly edging out Logistic Regression. MultinomialNB, while simpler, also provides competitive results.

**Chosen Model**: For the best performance and meeting the accuracy requirement, we will choose **SVM with TfidfVectorizer on lemmatized text** as our final model. It provides the highest accuracy and demonstrates the effectiveness of TF-IDF and lemmatization for this task.

In the next steps, we could further fine-tune the hyperparameters of the SVM model and potentially explore other advanced techniques. However, for now, we have a model that meets the project requirements. Let's proceed with finalizing our solution and preparing the necessary scripts and documentation.

## 5. Conclusion and Next Steps for MLE Part

We have successfully built a binary sentiment classification model that meets the required accuracy of 0.85. Our best model is SVM with TfidfVectorizer applied to lemmatized movie reviews. We've performed EDA, explored text preprocessing techniques, compared stemming and lemmatization, and tested different vectorization methods and models.

**Key Findings:**

*   TfidfVectorizer is slightly more effective than CountVectorizer for sentiment classification in our experiments.
*   Lemmatization provides a marginal improvement over stemming.
*   SVM and Logistic Regression achieved the best performance among the models tested.

**Next Steps (MLE Part):**

To complete the project, we need to focus on the MLE part:

1.  **Scripts**: We'll create separate Python scripts for data loading (`data_load.py`), training (`train.py`), and inference (`run_inference.py`). Modularize preprocessing, feature engineering, and model evaluation into separate functions.
2.  **Dockerfiles**: We'll create Dockerfiles for both training and inference environments to ensure reproducibility.
3.  **README.md**: We'll prepare a comprehensive README.md file that includes:
    *   DS part report: EDA conclusions, feature engineering description, model selection reasoning, performance evaluation, and potential business applications.
    *   ML part guide: Instructions on how to run the solution using Docker, quickstart guide.
4.  **requirements.txt**: We'll create a `requirements.txt` file listing all Python dependencies.
5.  **.gitignore**: We'll create a `.gitignore` file to exclude data, models, and other unnecessary files from Git.

We will now proceed to structure our project repository as per the provided example, create the necessary scripts, Dockerfiles, and documentation to finalize our solution. Let's start building the MLE components!